### No-show prediction

In [0]:
import pandas as pd

folder_name = '/Workspace/Users/asanders4205@gmail.com/no_show_prediction/noshows-prediction/input-datasets/'

dataset_name = "healthcare_noshows.csv"
dataset_path = f"{folder_name}{dataset_name}"



pdf = pd.read_csv(dataset_path)

In [0]:
access_df = spark.createDataFrame(pdf)

In [0]:
access_df.display()

%md
### Task 1.1: Load the Dataset

Load the CDC diabetes dataset from the provided path. Use the following options:
- `.option("nullValue", "null")` — read the string `"null"` as a SQL `null`
- `header="true"` — the CSV file includes a header row
- `inferSchema="true"` — let Spark automatically detect column data types
- `multiLine="true"` — handle multi-line CSV fields correctly

Once loaded, display the DataFrame to inspect its structure and column types.

### Specify the schema when reading in the file

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType

patient_schema = StructType([
    StructField('PatientId', IntegerType(), True),
    StructField('AppointmentID', IntegerType(), True),
    StructField('Gender', StringType(), True),
    StructField('ScheduledDay', DateType(), True),
    StructField('AppointmentDay', DateType(), True),
    StructField('Age', IntegerType(), True),
    StructField('Neighborhood', StringType(), True), # Renamed from british spelling 'Neighbourhood'
    StructField('Scholarship', BooleanType(), True),
    StructField('Hipertension', BooleanType(), True),
    StructField('Diabetes', BooleanType(), True),
    StructField('Alcoholism', BooleanType(), True),
    StructField('Handicap', BooleanType(), True),
    StructField('SMS_received', BooleanType(), True),
    StructField('Showed_up', BooleanType(), True),
    StructField('Date.diff', IntegerType(), True) 
])

In [0]:
''' Read in the csv'''
access_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("nullValue", "null") \
    .schema(patient_schema) \
    .load(dataset_path)


access_df = access_df.withColumnRenamed('Date.diff','date_diff')


%md
### Task 1.2: Data Preparation

With the data loaded, the next step is to prepare it for modeling. You will:
- **Cast data types** — Convert integer and boolean columns to `double` for compatibility with Spark ML
- **Remove columns with too many missing values** — Drop any column where more than 60% of values are missing
- **Remove outliers** — Filter records with invalid or extreme values
- **Save the cleaned data** — Write to a Delta silver table for reuse

In [0]:
from pyspark.sql.types import IntegerType, BooleanType
from pyspark.sql.functions import col

''' Prepare data for modelling
    Find columns with empty recoreds
    Remove rows with > 80% empty fields
'''
# access_df.printSchema()

# List integer and boolean columns - how to convert boolean and integer to double at once?
integer_cols = [
    c.name for c in access_df.schema.fields
    if isinstance(c.dataType, (IntegerType, BooleanType))
]

for column in integer_cols:
    access_df = access_df.withColumn(column, col(column).cast("double"))

# access_df.printSchema()

In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum

# Count missing values per column
missing_counts = access_df.agg(*[
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in access_df.columns
]).first().asDict()

# Display missing value counts as a summary DataFrame
missing_df = spark.createDataFrame(
    [(c, int(v)) for c, v in missing_counts.items()],
    ["column", "missing_count"]
)
display(missing_df.orderBy("missing_count", ascending=False))

In [0]:
access_df = access_df.drop('PatientId', 'AppointmentID')
display(access_df)

In [0]:
# [string_col.name for string_col in access_df.schema.fields if string_col.dataType.typeName() == "string"]

### Scale numeric values

In [0]:
# Base numeric features (cast to double in the cell above, IDs and target excluded)
numerical_cols = ["Age", "Scholarship", "Hipertension", "Diabetes", "Alcoholism", "Handicap", "SMS_received"]

### Handle Dates

In [0]:
import math
from pyspark.sql.functions import col, sin, cos, month, dayofweek, dayofyear, lit

access_df = (access_df
    .withColumn("Sched_month_sin",     sin(lit(2 * math.pi) * month(col("ScheduledDay"))      / 12))
    .withColumn("Sched_month_cos",     cos(lit(2 * math.pi) * month(col("ScheduledDay"))      / 12))
    .withColumn("Sched_dayofyear_sin", sin(lit(2 * math.pi) * dayofyear(col("ScheduledDay"))  / 365))
    .withColumn("Sched_dayofyear_cos", cos(lit(2 * math.pi) * dayofyear(col("ScheduledDay"))  / 365))
    .withColumn("Appoi_month_sin",     sin(lit(2 * math.pi) * month(col("AppointmentDay"))    / 12))
    .withColumn("Appoi_month_cos",     cos(lit(2 * math.pi) * month(col("AppointmentDay"))    / 12))
    .withColumn("Appoi_dayofyear_sin", sin(lit(2 * math.pi) * dayofyear(col("AppointmentDay"))/ 365))
    .withColumn("Appoi_dayofyear_cos", cos(lit(2 * math.pi) * dayofyear(col("AppointmentDay"))/ 365))
    .drop("ScheduledDay", "AppointmentDay")   # raw date columns not usable by VectorAssembler
)

# Add the cyclical columns to the feature list defined in the cell above
numerical_cols += [
    "Sched_month_sin", "Sched_month_cos", "Sched_dayofyear_sin", "Sched_dayofyear_cos",
    "Appoi_month_sin", "Appoi_month_cos", "Appoi_dayofyear_sin", "Appoi_dayofyear_cos",
]

access_df.select(*numerical_cols[-8:]).display()

In [0]:
train_df, test_df = access_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  Test: {test_df.count():,}")

In [0]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# Encode only the 'Gender' column
indexer = StringIndexer(inputCol="Gender", outputCol="Gender_index", handleInvalid="skip")
encoder = OneHotEncoder(inputCol="Gender_index", outputCol="Gender_vec")
vector_assembler = VectorAssembler(
    inputCols=["Gender_vec"] + numerical_cols,
    outputCol="features",
    handleInvalid="skip",
)

In [0]:
from sklearn.preprocessing import TargetEncoder
from pyspark.sql.functions import col, create_map, lit
from itertools import chain
import pandas as pd


# 1. Collect training data to pandas — fit encoder on train only
train_pd = train_df.select("Neighborhood", "Showed_up").toPandas()

enc = TargetEncoder(target_type="binary", smooth="auto")
enc.fit(train_pd[["Neighborhood"]], train_pd["Showed_up"])

# 2. Build a lookup map: Neighborhood string → encoded float
categories   = enc.categories_[0]                  # array of Neighborhood names
encoded_vals = enc.transform(pd.DataFrame({"Neighborhood": categories}))

mapping = dict(zip(categories, encoded_vals[:, 0].tolist()))

# 3. Apply the map to both splits as a new Spark column
map_expr = create_map([lit(x) for x in chain.from_iterable(mapping.items())])

train_df = train_df.withColumn("Neighborhood_te", map_expr[col("Neighborhood")])
test_df  = test_df.withColumn("Neighborhood_te", map_expr[col("Neighborhood")])

# 4. Add to your numerical features and drop the raw string column
numerical_cols += ["Neighborhood_te"]
train_df = train_df.drop("Neighborhood")
test_df  = test_df.drop("Neighborhood")


In [0]:
pipeline = Pipeline(stages=[indexer, encoder, vector_assembler])

model = pipeline.fit(train_df)
train_encoded = model.transform(train_df)
test_encoded = model.transform(test_df)

display(train_encoded.select("features"))

In [0]:
%skip
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

categorical_cols = ["Gender", "Neighborhood"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="skip")
    for c in categorical_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_index", outputCol=f"{c}_vec")
    for c in categorical_cols
]
vector_assembler = VectorAssembler(
    inputCols=[f"{c}_vec" for c in categorical_cols] + numerical_cols,
    outputCol="features",
    handleInvalid="skip",
)

pipeline = Pipeline(stages=indexers + encoders + [vector_assembler])

# Fit on training data only — prevents test-set statistics leaking into the pipeline
model        = pipeline.fit(train_df)
train_encoded = model.transform(train_df)
test_encoded  = model.transform(test_df)

train_encoded.select("features").show(5)

In [0]:
train_encoded.printSchema()

In [0]:
# Drop string columns that were encoded — the encoded _vec columns are already in the feature vector
train_encoded = train_encoded.drop('Gender', 'Neighborhood')
test_encoded  = test_encoded.drop('Gender', 'Neighborhood')
display(train_encoded)